In [3]:
import pandas as pd
from IPython.display import display, Markdown

aic = pd.read_csv("regression_output/all_rows/_collected/all_aic.csv")
anova = pd.read_csv("regression_output/all_rows/_collected/all_anova.csv")
coef = pd.read_csv("regression_output/all_rows/_collected/all_coefficients.csv")

pretty_map = {
    "gpt2_uni": "GPT-2",
    "gemma270m_uni": "Gemma-270M",
    "gemma12b_uni": "Gemma-12B",
    "swp_all": "SwP",
    "swp_target_only": "SwP Present Target"
}

aic_main = aic[
    (aic["dataset_label"] == "filtered_17k") &
    (aic["group_name"] == "primary_uni")
].copy()

anova_main = anova[
    (anova["dataset_label"] == "filtered_17k") &
    (anova["group_name"] == "primary_uni")
].copy()

coef_main = coef[
    (coef["dataset_label"] == "filtered_17k") &
    (coef["group_name"] == "primary_uni")
].copy()

aic_wide = (
    aic_main.pivot_table(index="predictor_label", columns="model", values="AIC")
    .reset_index()
)

anova_keep = anova_main[[
    "predictor_label", "comparison", "Chisq", "p_value"
]].copy()

anova_wide = anova_keep.pivot_table(
    index="predictor_label",
    columns="comparison",
    values=["Chisq", "p_value"]
)

anova_wide.columns = [f"{a}_{b}" for a, b in anova_wide.columns]
anova_wide = anova_wide.reset_index()

coef_pred = coef_main[
    (coef_main["model"] == "m_pred") &
    (~coef_main["term"].isin(["(Intercept)", "word_length", "word_frequency"]))
][["predictor_label", "term", "estimate", "p_value"]].copy()

coef_pred = coef_pred.rename(columns={
    "term": "predictor_term",
    "estimate": "pred_beta",
    "p_value": "pred_beta_p"
})

benchmark_table = (
    aic_wide
    .merge(anova_wide, on="predictor_label", how="left")
    .merge(coef_pred, on="predictor_label", how="left")
)

benchmark_table["Predictor"] = benchmark_table["predictor_label"].map(pretty_map).fillna(benchmark_table["predictor_label"])

benchmark_table = benchmark_table.rename(columns={
    "m_base": "AIC_base",
    "m_cloze": "AIC_cloze",
    "m_pred": "AIC_pred",
    "m_both": "AIC_both",
    "Chisq_predictor_vs_base": "LR_pred_vs_base",
    "p_value_predictor_vs_base": "p_pred_vs_base",
    "Chisq_predictor_beyond_cloze": "LR_pred_beyond_cloze",
    "p_value_predictor_beyond_cloze": "p_pred_beyond_cloze",
    "Chisq_cloze_beyond_predictor": "LR_cloze_beyond_pred",
    "p_value_cloze_beyond_predictor": "p_cloze_beyond_pred"
})

benchmark_table = benchmark_table[[
    "Predictor",
    "AIC_pred",
    "AIC_both",
    "LR_pred_vs_base",
    "p_pred_vs_base",
    "LR_pred_beyond_cloze",
    "p_pred_beyond_cloze",
    "LR_cloze_beyond_pred",
    "p_cloze_beyond_pred",
    "pred_beta",
    "pred_beta_p"
]]

display(benchmark_table.round(3))

,Predictor,AIC_pred,AIC_both,LR_pred_vs_base,p_pred_vs_base,LR_pred_beyond_cloze,p_pred_beyond_cloze,LR_cloze_beyond_pred,p_cloze_beyond_pred,pred_beta,pred_beta_p
0,Gemma-12B,1192.588,1193.822,54.058,0.0,7.412,0.006,0.766,0.381,0.002,0.0
1,Gemma-270M,1197.434,1195.680,49.211,0.0,5.554,0.018,3.754,0.053,0.002,0.0
2,GPT-2,1198.159,1194.108,48.487,0.0,7.126,0.008,6.051,0.014,0.003,0.0
3,swp,1224.214,1201.118,22.432,0.0,0.116,0.733,25.097,0.000,0.001,0.0
4,SwP Present Target,944.664,935.027,17.778,0.0,0.018,0.893,11.637,0.001,0.001,0.0


In [2]:
import pandas as pd

aic = pd.read_csv("regression_output/coverage_controlled/_collected/all_aic.csv")
anova = pd.read_csv("regression_output/coverage_controlled/_collected/all_anova.csv")
coef = pd.read_csv("regression_output/coverage_controlled/_collected/all_coefficients.csv")

pred_only = aic[aic["model"] == "m_pred"][["predictor_label", "AIC"]].rename(columns={"AIC": "AIC_pred"})
both = aic[aic["model"] == "m_both"][["predictor_label", "AIC"]].rename(columns={"AIC": "AIC_both"})

pred_vs_base = anova[anova["comparison"] == "predictor_vs_base"][["predictor_label", "Chisq", "p_value"]]
pred_vs_base = pred_vs_base.rename(columns={"Chisq": "LR_pred_vs_base", "p_value": "p_pred_vs_base"})

pred_beyond_cloze = anova[anova["comparison"] == "predictor_beyond_cloze"][["predictor_label", "Chisq", "p_value"]]
pred_beyond_cloze = pred_beyond_cloze.rename(columns={"Chisq": "LR_pred_beyond_cloze", "p_value": "p_pred_beyond_cloze"})

cloze_beyond_pred = anova[anova["comparison"] == "cloze_beyond_predictor"][["predictor_label", "Chisq", "p_value"]]
cloze_beyond_pred = cloze_beyond_pred.rename(columns={"Chisq": "LR_cloze_beyond_pred", "p_value": "p_cloze_beyond_pred"})

pred_beta = coef[(coef["model"] == "m_pred") & (~coef["term"].isin(["(Intercept)", "word_length", "word_frequency"]))][["predictor_label", "estimate", "p_value"]]
pred_beta = pred_beta.rename(columns={"estimate": "pred_beta", "p_value": "pred_beta_p"})

summary = (
    pred_only
    .merge(both, on="predictor_label")
    .merge(pred_vs_base, on="predictor_label")
    .merge(pred_beyond_cloze, on="predictor_label")
    .merge(cloze_beyond_pred, on="predictor_label")
    .merge(pred_beta, on="predictor_label")
    .sort_values("AIC_pred")
)

display(summary.round(3))

,predictor_label,AIC_pred,AIC_both,LR_pred_vs_base,p_pred_vs_base,LR_pred_beyond_cloze,p_pred_beyond_cloze,LR_cloze_beyond_pred,p_cloze_beyond_pred,pred_beta,pred_beta_p
0,gemma12b_uni,927.675,929.474,34.767,0.0,5.571,0.018,0.201,0.654,0.003,0.0
2,gpt2_uni,933.809,931.912,28.634,0.0,3.133,0.077,3.897,0.048,0.003,0.0
1,gemma270m_uni,935.192,933.683,27.250,0.0,1.363,0.243,3.509,0.061,0.002,0.0
3,swp,944.664,935.027,17.778,0.0,0.018,0.893,11.637,0.001,0.001,0.0
